# M3KG-RAG reproduction (scaled-down, AudioCaps-QA) — Colab driver
Runs `scripts/m3kg/pipeline.sh` stage by stage. *Runtime → Run all* is safe: every cell stops at the first failed command, every stage is resumable (all model calls are cached), and results are copied to Google Drive after each stage.

### 0. Session setup
Runtime → *Change runtime type* → **L4 GPU** (Colab Pro, recommended). The `colab_l4` profile (Qwen3-8B agents, Qwen2.5-Omni-7B, Llama-3-8B-AWQ judge) is used on every GPU with at least 20 GB, so results do not depend on the GPU; a 16 GB T4 switches to the `colab_t4` profile (4-bit agents, Qwen2.5-Omni-3B).

Keep this browser tab open while it runs (Colab Pro has no background execution). If the session ends, open the notebook again and *Run all*: finished work is restored from Drive and skipped.

Store `WANDB_API_KEY` and `HF_TOKEN` in Colab *Secrets* (key icon on the left) and enable *Notebook access* for both. Never paste keys into the notebook: the repository is public.

In [ ]:
from google.colab import drive, userdata
import os
drive.mount('/content/drive')
for k in ('WANDB_API_KEY', 'HF_TOKEN'):
    try:
        os.environ[k] = userdata.get(k)
    except Exception:
        print('secret not set:', k)
# W&B runs go to the default entity of WANDB_API_KEY; set WANDB_ENTITY to use a team instead.
os.environ['ENV_ROOT'] = '/content/envs'

import codecs, signal, subprocess, sys

def sh(cmd, check=True):
    """Run a shell command and stream its output. A failing command raises an error,
    so 'Run all' stops at the first failed stage instead of running the next ones."""
    p = subprocess.Popen(cmd, shell=True, executable='/bin/bash', stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, start_new_session=True)
    dec = codecs.getincrementaldecoder('utf-8')('replace')
    try:
        while True:
            b = os.read(p.stdout.fileno(), 4096)
            if not b:
                break
            sys.stdout.write(dec.decode(b)); sys.stdout.flush()
        rc = p.wait()
    except KeyboardInterrupt:          # 'Interrupt execution' also stops the GPU job
        os.killpg(p.pid, signal.SIGINT)
        try:
            p.wait(timeout=30)
        except subprocess.TimeoutExpired:
            os.killpg(p.pid, signal.SIGKILL)
        raise
    if rc != 0 and check:
        raise RuntimeError(f'command failed (exit code {rc}): {cmd}')

In [ ]:
%cd /content
sh('git clone https://github.com/NoiceSmoker/mmgraphrag-repro.git 2>/dev/null || (cd mmgraphrag-repro && git pull)')
%cd /content/mmgraphrag-repro
smi = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader,nounits'],
                     capture_output=True, text=True)
if smi.returncode != 0:
    raise RuntimeError('No GPU in this session: Runtime -> Change runtime type -> L4 GPU')
gpu_name, gpu_mib = smi.stdout.strip().split('\n')[0].rsplit(', ', 1)
gpu_mib = int(float(gpu_mib))
os.environ['PROFILE'] = ('configs/m3kg/profiles/colab_t4.yaml' if gpu_mib < 20000
                         else 'configs/m3kg/profiles/colab_l4.yaml')
print(f"{gpu_name} ({gpu_mib} MiB) -> {os.environ['PROFILE']}")

### 1. Environments (once per VM)

In [ ]:
sh('bash scripts/setup_envs.sh vllm m3kg-audio')

### 2. Data: AudioCaps-QA test set + AudioCaps train subset (corpus)

In [ ]:
sh('bash scripts/colab/sync.sh pull m3kg')
sh('bash scripts/m3kg/pipeline.sh data && bash scripts/colab/sync.sh push m3kg')

### 3. (optional) YouTube metadata for the rewriter agent
Allowed to fail (removed videos, rate limits); the `kg` stage uses whatever metadata it finds.

In [ ]:
sh('bash scripts/m3kg/pipeline.sh youtube && bash scripts/colab/sync.sh push m3kg', check=False)

### 4. Multi-agent KG construction (Qwen3 + Wikipedia/Wiktionary)

In [ ]:
sh("WIKI_USER_AGENT='mmgraphrag-repro/0.1 (https://github.com/NoiceSmoker/mmgraphrag-repro)' "
   "bash scripts/m3kg/pipeline.sh kg && bash scripts/colab/sync.sh push m3kg")

### 5. CLAP index, retrieval + TAG grounding for all configurations

In [ ]:
sh('bash scripts/m3kg/pipeline.sh index && bash scripts/colab/sync.sh push m3kg')
sh('bash scripts/m3kg/pipeline.sh contexts && bash scripts/colab/sync.sh push m3kg')

### 6. GRASP LLM filter → Qwen2.5-Omni generation → AudioBench judge

In [ ]:
sh('bash scripts/m3kg/pipeline.sh filter && bash scripts/colab/sync.sh push m3kg')
sh('bash scripts/m3kg/pipeline.sh generate && bash scripts/colab/sync.sh push m3kg')
sh('bash scripts/m3kg/pipeline.sh judge && bash scripts/colab/sync.sh push m3kg')

### 7. Pairwise win-rate judge, then metrics → W&B

In [ ]:
sh('bash scripts/m3kg/pipeline.sh winrate && bash scripts/colab/sync.sh push m3kg')
sh('bash scripts/m3kg/pipeline.sh report && bash scripts/colab/sync.sh push m3kg')

### 8. W&B report pages + LaTeX tables
Links of both papers are collected in `results/wandb_links.json` on Drive; the LaTeX fragments in `results/latex` are complete once both notebooks have finished.

In [ ]:
sh('bash scripts/colab/sync.sh pull results')
sh('/content/envs/m3kg-audio/bin/python scripts/make_wandb_reports.py --only m3kg --share')
sh('/content/envs/m3kg-audio/bin/python scripts/export_results.py --results results --out results/latex')
sh('bash scripts/colab/sync.sh push results')